# Modeling

In [221]:
import pandas as pd
from sklearn.decomposition import PCA
import numpy as np

from sklearn.linear_model import Ridge, Lasso, ElasticNet, BayesianRidge, HuberRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.ensemble import (
    GradientBoostingRegressor,
    RandomForestRegressor,
    ExtraTreesRegressor
)

import xgboost as xgb
from sklearn.pipeline import Pipeline
from sklearn import model_selection



In [222]:
df_train = pd.read_csv('../data/data_model/train_model.csv', sep=',')
df_test = pd.read_csv('../data/data_model/test_model.csv', sep=',')

In [223]:
X_train = df_train.drop(columns=['SALEPRICE'])
y_train = df_train['SALEPRICE']

In [224]:
df_train.head()

,NEIGHBORHOOD,OVERALLQUAL,EXTERQUAL,BSMTQUAL,HEATINGQC,GRLIVAREA,KITCHENQUAL,TOTRMSABVGRD,GARAGEYRBLT,GARAGEFINISH,...,EXTERIOR1ST_OTHER,EXTERIOR1ST_PLYWOOD,EXTERIOR1ST_VINYLSD,EXTERIOR1ST_WD SDNG,GARAGETYPE_DETCHD,GARAGETYPE_OTHER,SALETYPE_CON,SALETYPE_NEW,SALETYPE_WD,SALEPRICE
0,0.483681,0.550585,0.930835,0.491996,0.814688,0.531594,0.636941,0.969273,0.916845,0.123461,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,12.247699
1,1.142755,-0.233625,-0.827054,0.491996,0.814688,-0.472592,-0.935418,-0.338533,-0.282761,0.123461,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,12.109016
2,0.483681,0.550585,0.930835,0.491996,0.814688,0.675348,0.636941,-0.338533,0.827986,0.123461,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,12.317171
3,1.801830,1.334796,0.930835,0.491996,0.814688,1.361584,0.636941,1.623175,0.783556,0.123461,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,12.429220
4,0.977986,1.334796,0.930835,1.996684,0.814688,0.500517,0.636941,0.315370,0.961275,0.123461,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,12.634606


In [225]:
pca_analysis = PCA(random_state=0) 
pca_analysis.fit(X_train)

individual_variance = pca_analysis.explained_variance_ratio_
accumulated_variance = individual_variance.cumsum()

pca_table = pd.DataFrame({
    'COMPONENT': range(1, len(accumulated_variance) + 1),
    'INDIVIDUAL_VARIANCE': (individual_variance).round(4),
    'ACCUMULATED_VARIANCE': (accumulated_variance).round(4),
    'DELTA': (pd.Series(accumulated_variance).diff().fillna(accumulated_variance[0]) * 100).round(4)
})

first_above_95 = np.argmax(np.array(pca_table['ACCUMULATED_VARIANCE']) >= 0.95)
first_above_95

np.int64(16)

In [226]:
pca = PCA(n_components=16, random_state=0)
X_pca = pca.fit_transform(X_train) 

In [227]:
models = {
    'RIDGE':                   Ridge(),
    'LASSO':                   Lasso(),
    'ELASTICNET':              ElasticNet(),
    'BAYESIAN RIDGE':          BayesianRidge(),
    'HUBER':                   HuberRegressor(max_iter=1000),
    'KNN REGRESSOR':           KNeighborsRegressor(),
    'SVR':                     SVR(),
    'RANDOM FOREST':           RandomForestRegressor(random_state=0),
    'EXTRA TREES':             ExtraTreesRegressor(random_state=0),
    'GRADIENT BOOSTING':       GradientBoostingRegressor(random_state=0),
    'XGBOOST':                 xgb.XGBRegressor(random_state=0)
}

kfold = model_selection.KFold(
    n_splits=10,
    shuffle=True,
    random_state=0
)

results = []

for name, model in models.items():

    pipeline = Pipeline([
        ('model', model)
    ])

    scores = model_selection.cross_validate(
        pipeline,
        X_train,
        y_train,
        scoring=['r2', 'neg_mean_absolute_error', 'neg_mean_squared_error'],
        cv=kfold,
        n_jobs=-1
    )

    r2   = round(scores['test_r2'].mean(), 4)
    mae  = round(-scores['test_neg_mean_absolute_error'].mean(), 4)
    rmse = round(np.sqrt(-scores['test_neg_mean_squared_error']).mean(), 4)

    result = {
        'name': name,
        'r2':   r2,
        'mae':  mae,
        'rmse': rmse
    }
    results.append(result)

In [228]:
df_exploracao = pd.DataFrame(results)
df_exploracao.columns = df_exploracao.columns.str.upper()
df_exploracao = df_exploracao.round(4)

df_exploracao.sort_values(by=['R2'], ascending=False)

,NAME,R2,MAE,RMSE
9,GRADIENT BOOSTING,0.8782,0.0868,0.1196
8,EXTRA TREES,0.8779,0.0848,0.1200
7,RANDOM FOREST,0.8725,0.0878,0.1223
0,RIDGE,0.8702,0.0928,0.1236
3,BAYESIAN RIDGE,0.8699,0.0927,0.1237
6,SVR,0.8694,0.0888,0.1239
4,HUBER,0.8688,0.0918,0.1242
10,XGBOOST,0.8680,0.0896,0.1247
5,KNN REGRESSOR,0.8384,0.0988,0.1377
2,ELASTICNET,-0.0123,0.2744,0.3455


In [230]:
df_exploracao_analysis = df_exploracao.copy()
df_exploracao_analysis[['MAE', 'RMSE']] = df_exploracao_analysis[['MAE', 'RMSE']].rank(0, numeric_only=True, ascending=True)
df_exploracao_analysis['R2'] = df_exploracao_analysis['R2'].rank(0, numeric_only=True, ascending=False)

df_exploracao_analysis['POINTS'] = df_exploracao_analysis['R2'] +  df_exploracao_analysis['MAE'] + df_exploracao_analysis['RMSE'] 
df_exploracao_analysis = df_exploracao_analysis.sort_values(by=['POINTS'], ascending=True)
df_exploracao_analysis = df_exploracao_analysis.reset_index(drop=True)

df_exploracao_analysis


,NAME,R2,MAE,RMSE,POINTS
0,GRADIENT BOOSTING,1.0,2.0,1.0,4.0
1,EXTRA TREES,2.0,1.0,2.0,5.0
2,RANDOM FOREST,3.0,3.0,3.0,9.0
3,RIDGE,4.0,8.0,4.0,16.0
4,SVR,6.0,4.0,6.0,16.0
5,BAYESIAN RIDGE,5.0,7.0,5.0,17.0
6,HUBER,7.0,6.0,7.0,20.0
7,XGBOOST,8.0,5.0,8.0,21.0
8,KNN REGRESSOR,9.0,9.0,9.0,27.0
9,ELASTICNET,10.5,10.5,10.5,31.5
